In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType

In [0]:
df = spark.table("workspace.bronze.sellers")


In [0]:
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

df = df.select(
    *[F.trim(F.col(c)).alias(c) if c in string_cols else F.col(c) for c in df.columns]
)

In [0]:
df = (
    df
    .filter(F.col("seller_id").isNotNull())
    
    .dropDuplicates(["seller_id"])
    
    .withColumn("seller_city", F.lower(F.coalesce(F.col("seller_city"), F.lit("unknown"))))
    
    .withColumn("seller_state", F.upper(F.col("seller_state")))
    
    .withColumnRenamed("seller_zip_code_prefix", "zip_prefix")
    
    .withColumn(
        "zip_prefix",
        F.coalesce(F.col("zip_prefix").cast(IntegerType()), F.lit(0))
    )
    
    .withColumn("silver_processed_time", F.current_timestamp())
)

In [0]:
df.limit(5).display()

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.sellers")
)